## 00 — The Viewer

This is the assembly notebook. Every component we built across Modules 00–05 is brought together here into one clean, readable file.

No new concepts. The goal is:
- See all the pieces side by side
- Produce a working viewer with styled output
- Understand the full data flow in one place

Read through the code before running it. Every line should be recognizable.

## 1. Imports and Paths

In [1]:
import json
import math
import time
from pathlib import Path
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

LOD_DIR = Path("../../data/lod")

LOD_CONFIG = [
    {"name": "coarse",     "file": "railroads_coarse.geojson",     "zoom_max": 3},
    {"name": "medium",     "file": "railroads_medium.geojson",     "zoom_max": 6},
    {"name": "fine",       "file": "railroads_fine.geojson",       "zoom_max": 10},
    {"name": "extra_fine", "file": "railroads_extra_fine.geojson", "zoom_max": 99},
]

## 2. Geometry and Index Utilities

In [2]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


def leaflet_bounds_to_bbox(bounds):
    """Convert ipyleaflet [[lat_min,lon_min],[lat_max,lon_max]] to [lon_min,lat_min,lon_max,lat_max]."""
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]


class GridIndex:
    """Uniform grid spatial index. Assigns features to 10° cells for fast viewport queries."""

    CELL_SIZE = 10.0

    def __init__(self):
        self.cells = {}

    def _cells(self, bbox):
        lon_min, lat_min, lon_max, lat_max = bbox
        cs = self.CELL_SIZE
        col_min, col_max = int((lon_min + 180) / cs), int((lon_max + 180) / cs)
        row_min, row_max = int((lat_min +  90) / cs), int((lat_max +  90) / cs)
        return [(c, r) for c in range(col_min, col_max + 1) for r in range(row_min, row_max + 1)]

    def build(self, features):
        self.cells = {}
        for idx, f in enumerate(features):
            for cell in self._cells(feature_bbox(f)):
                self.cells.setdefault(cell, []).append((idx, f))

    def query(self, viewport_bbox):
        seen, results = set(), []
        for cell in self._cells(viewport_bbox):
            for idx, f in self.cells.get(cell, []):
                if idx not in seen:
                    seen.add(idx)
                    results.append(f)
        return results

## 3. Load Data and Build Indexes

In [3]:
lod_indexes = {}

for cfg in LOD_CONFIG:
    t0 = time.perf_counter()
    with open(LOD_DIR / cfg["file"]) as f:
        features = json.load(f)["features"]
    idx = GridIndex()
    idx.build(features)
    elapsed = time.perf_counter() - t0
    lod_indexes[cfg["name"]] = idx
    print(f"  {cfg['name']:<12}  {len(features):>6,} features  {elapsed:.2f}s")

print("\nReady.")

  coarse        25,413 features  0.20s
  medium        25,413 features  0.28s
  fine          25,413 features  0.50s
  extra_fine    25,413 features  0.48s

Ready.


## 4. LOD Selection

In [4]:
def get_lod(zoom):
    z = int(math.floor(zoom))
    for cfg in LOD_CONFIG:
        if z <= cfg["zoom_max"]:
            return cfg["name"]
    return LOD_CONFIG[-1]["name"]

## 5. Styling

We use `scalerank` to vary line weight — more important lines render thicker, just like a real map.

In [5]:
def style_callback(feature):
    rank = feature["properties"].get("scalerank", 5)
    weight = max(0.5, 2.5 - rank * 0.3)   # rank 1 → thick, rank 7 → thin
    return {
        "color":   "#333333",
        "weight":  weight,
        "opacity": 0.85,
    }

## 6. The Map

In [6]:
current_lod = get_lod(5)

m = Map(center=[48.5, 10.0], zoom=5)

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style_callback=style_callback
)
m.add(layer)

# Status bar
lbl_lod      = widgets.Label()
lbl_zoom     = widgets.Label()
lbl_features = widgets.Label()
lbl_time     = widgets.Label()
status       = widgets.HBox([lbl_lod, lbl_zoom, lbl_features, lbl_time])

def update(*args):
    global current_lod
    if not m.bounds:
        return

    current_lod    = get_lod(m.zoom)
    vp             = leaflet_bounds_to_bbox(m.bounds)

    t0             = time.perf_counter()
    visible        = lod_indexes[current_lod].query(vp)
    elapsed_ms     = (time.perf_counter() - t0) * 1000

    layer.data          = {"type": "FeatureCollection", "features": visible}
    lbl_lod.value       = f"LOD: {current_lod}"
    lbl_zoom.value      = f"  zoom: {int(math.floor(m.zoom))}"
    lbl_features.value  = f"  features: {len(visible):,}"
    lbl_time.value      = f"  query: {elapsed_ms:.1f}ms"

m.observe(update, names=["zoom", "bounds"])
update()

widgets.VBox([m, status])

## Exercise A

Add a second style dimension: color the lines by `category` property.

1. Find the unique `category` values in the fine LOD file
2. Assign a distinct color to each
3. Update `style_callback` to use category color + scalerank weight together

The result should show the railroad network colored by line type.

In [ ]:
# Add category-based coloring to style_callback
# Your code here

In [1]:

import json
from ipyleaflet import Map, GeoJSON, FullScreenControl
import math
import time
from pathlib import Path
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

data_path = Path("../../data/ne_10m_railroads.geojson")

with open(data_path) as f:
    railroads = json.load(f)
LOD_DIR = Path("../../data/lod")

LOD_CONFIG = [
    {"name": "coarse",     "file": "railroads_coarse.geojson",     "zoom_max": 3},
    {"name": "medium",     "file": "railroads_medium.geojson",     "zoom_max": 6},
    {"name": "fine",       "file": "railroads_fine.geojson",       "zoom_max": 10},
    {"name": "extra_fine", "file": "railroads_extra_fine.geojson", "zoom_max": 99},
]

# Find unique categories and map them to distinct colors
unique_categories = [feature['properties'].get('category') for feature in railroads['features'] if feature['properties'].get('category')]
unique_categories = list(set(unique_categories))

# Create a color map (e.g., using matplotlib cm or manual list)
color_palette = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33']
category_colors = {
    category: color_palette[i % len(color_palette)] 
    for i, category in enumerate(unique_categories)
}
print(f"Categories mapped: {category_colors}")

# 2. Assign unique colors (This logic is usually handled within the callback,
# but can be pre-calculated for efficiency)

# 3. Define the style_callback to use category color + scalerank weight
def style_callback(feature):
    """
    Sets style: Color by 'category', Weight by 'scalerank'
    """
    # Default values
    color = '#666666'
    weight = 1
    
    # Apply category color
    category = feature['properties'].get('category')
    if category in category_colors:
        color = category_colors[category]
        
    # Apply weight based on scalerank (assuming lower scalerank = more important/thicker)
    scalerank = feature['properties'].get('scalerank', 5)
    weight = max(1, 6 - int(scalerank)) # Example: rank 1-5 maps to weight 5-1
    
    return {
        'color': color,
        'weight': weight,
        'opacity': 0.8
    }

# Create the Map
m = Map(center=(39.8, -98.5), zoom=4)

# Create and add the GeoJSON layer
railroad_layer = GeoJSON(
    data=railroads,
    style_callback=style_callback,
    hover_style={'color': 'white', 'weight': 5}
)

m.add_layer(railroad_layer)
m.add_control(FullScreenControl())

# Display the map
m


Categories mapped: {1: '#e41a1c', 2: '#377eb8', 3: '#4daf4a', 6: '#984ea3', 7: '#ff7f00', 9: '#ffff33'}


Map(center=[39.8, -98.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## Exercise B

Add a tooltip: when the user hovers over a railroad feature, show its `category` and `scalerank` in a widget below the map.

Hint: use the GeoJSON layer's `on_hover` event.

In [ ]:
# Add hover tooltip showing category and scalerank
# Your code here

In [ ]:
from ipyleaflet import Map, GeoJSON, WidgetControl
from ipywidgets import HTML
import json
data_path = Path("../../data/ne_10m_railroads.geojson")

with open(data_path) as f:
    railroads = json.load(f)
# 1. Create the HTML widget for displaying info below the map
info_widget = HTML(
    value='<h4>Hover over a railroad</h4>',
    layout={'border': '1px solid black', 'padding': '5px', 'background': 'white'}
)

# 2. Set up the Map
m = Map(center=(40, -100), zoom=4)

# 3. Define the hover callback function
def update_info(feature, **kwargs):
    props = feature['properties']
    # Customize the text displayed in the widget
    info_widget.value = f"""
        <h4>Railroad Info</h4>
        <b>Category:</b> {props.get('category', 'N/A')}<br>
        <b>Scalerank:</b> {props.get('scalerank', 'N/A')}
    """


geojson_layer = GeoJSON(
    data=railroads,
    style={'color': 'blue', 'weight': 3},
    hover_style={'color': 'red', 'weight': 5}
)

# 5. Attach the event handler
geojson_layer.on_hover(update_info)

# Add layers to map
m.add_layer(geojson_layer)

# Display the map and widget
from ipywidgets import VBox
display(VBox([m, info_widget]))


## Check Your Understanding

The viewer re-runs `get_lod()` and queries the grid on **every** bounds change — even if the user only pans a few pixels and the LOD hasn't changed.

Describe two optimizations you could apply to skip redundant work:
1. One that avoids calling `get_lod()` unnecessarily
2. One that avoids re-querying the grid when the viewport barely moved

For each, what is the tradeoff?

---

We can implement the following optimizations to avoid redundant work:
*Avoid calling `get_lod()` unnecessarily: One of the optimization might be Cache/Check  when LOD is has Changed. Instead of immediately recalculating the LOD, store the result of the last get_lod() calculation along with the zoom level or view scale. On every bounds change, first check if the zoom/scale level has actually changed enough to require a new LOD. The tradeoff will be keeping the LOD as it is if the LOD threshold calculation is based on more than just the camera zoom. It might potentially showing a slightly lower-resolution LOD for a few frames after a quick zoom. 

*Avoids re-querying the grid when the viewport barely moved:To avoid unnecessary grid refreshes when the viewport barely moves, the system can update the grid only after the pan exceeds a defined pixel threshold (hysteresis) or after a short delay once panning stops (debounce). This reduces CPU load and makes panning feel smoother. The tradeoff is that the grid edges may momentarily lag or flicker when new data finally loads, and rapid panning can introduce a noticeable delay before fresh data appears.


## Next

In [01 — What We Built](./01-What_We_Built.ipynb), we step back, measure the system's remaining limitations, and document every decision we made.